<a href="https://colab.research.google.com/github/athishsreeram/tubenotebook/blob/main/Transcript_For_Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install youtube-transcript-api --upgrade
# 87Pm0SGTtN8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 15.7 MB/s eta 0:00:00


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
import re
import requests
from urllib.parse import urlparse, parse_qs

def get_video_info(video_id):
    """
    Fetch video title and other metadata using YouTube's oEmbed API.

    Args:
        video_id (str): The YouTube video ID.

    Returns:
        dict: {
            "title": str,
            "url": str
        }
    """
    try:
        # Using YouTube's oEmbed endpoint to get video metadata
        oembed_url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
        response = requests.get(oembed_url)
        response.raise_for_status()
        data = response.json()

        return {
            "title": data.get("title", f"Video_{video_id}"),
            "url": f"https://www.youtube.com/watch?v={video_id}"
        }
    except Exception as e:
        print(f"⚠️ Could not fetch video info for {video_id}: {e}")
        return {
            "title": f"Video_{video_id}",
            "url": f"https://www.youtube.com/watch?v={video_id}"
        }

def sanitize_filename(filename):
    """
    Remove invalid characters from filename.

    Args:
        filename (str): Original filename.

    Returns:
        str: Sanitized filename safe for file system.
    """
    # Remove invalid characters for Windows/Linux/Mac
    filename = re.sub(r'[<>:"/\\|?*]', '', filename)
    # Remove leading/trailing spaces and dots
    filename = filename.strip('. ')
    # Limit length to 200 characters
    if len(filename) > 200:
        filename = filename[:200]
    return filename

def get_transcript(video_id, langs=None):
    """
    Fetch transcript for a YouTube video.

    Args:
        video_id (str): The YouTube video ID.
        langs (list, optional): List of language codes to try, e.g. ['en','hi'].
                                Defaults to ['en'].

    Returns:
        dict: {
            "video_id": str,
            "video_url": str,
            "video_title": str,
            "transcript": str,
            "segments": list of {"text": str, "start": float, "duration": float}
        }
        or None if no transcript available.
    """
    if langs is None:
        langs = ["en"]

    try:
        # Get video info first
        video_info = get_video_info(video_id)

        # Fetch transcript
        transcript = YouTubeTranscriptApi().fetch(video_id, languages=langs)
        full_text = " ".join([t.text for t in transcript])
        segments = [{"text": t.text, "start": t.start, "duration": t.duration} for t in transcript]

        return {
            "video_id": video_id,
            "video_url": video_info["url"],
            "video_title": video_info["title"],
            "transcript": full_text,
            "segments": segments
        }
    except Exception as e:
        print(f"⚠️ Transcript unavailable for {video_id}: {e}")
        return None

def save_transcript_to_md(data, output_dir="."):
    """
    Save transcript data to a Markdown file named after the video title.

    Args:
        data (dict): The transcript data from get_transcript()
        output_dir (str): Directory to save the markdown file. Defaults to current directory.

    Returns:
        str: Path to the saved file or None if failed.
    """
    if not data:
        print("❌ No data to save")
        return None

    # Create valid filename from video title
    base_filename = sanitize_filename(data["video_title"])
    md_filename = f"{base_filename}.md"

    # Add directory path
    if output_dir != ".":
        import os
        os.makedirs(output_dir, exist_ok=True)
        md_filepath = os.path.join(output_dir, md_filename)
    else:
        md_filepath = md_filename

    # Create markdown content
    md_content = f"""# {data["video_title"]}

## Video Information
- **Video URL:** {data["video_url"]}
- **Video ID:** {data["video_id"]}

## Transcript

{data["transcript"]}

## Detailed Segments

"""

    # Add timestamped segments
    for i, segment in enumerate(data["segments"], 1):
        minutes = int(segment["start"] // 60)
        seconds = int(segment["start"] % 60)
        timestamp = f"{minutes:02d}:{seconds:02d}"

        md_content += f"### Segment {i} [{timestamp}]\n"
        md_content += f"**Duration:** {segment['duration']:.1f} seconds\n\n"
        md_content += f"{segment['text']}\n\n"
        md_content += "---\n\n"

    # Write to file
    try:
        with open(md_filepath, "w", encoding="utf-8") as f:
            f.write(md_content)
        print(f"✅ Transcript saved to: {md_filepath}")
        return md_filepath
    except Exception as e:
        print(f"❌ Error saving file: {e}")
        return None

# 🔥 Example usage
if __name__ == "__main__":
    video_id = "hBUTXCAc5zg"  # Replace with any video ID
    data = get_transcript(video_id, langs=["en", "hi"])  # Try English and Hindi

    if data:
        print("✅ Transcript fetched successfully!")
        print(f"📹 Video Title: {data['video_title']}")
        print(f"🔗 Video URL: {data['video_url']}")
        print(f"📝 Transcript Preview: {data['transcript'][:200]}...")

        # Save to Markdown file
        save_transcript_to_md(data, output_dir="transcripts")
    else:
        print("❌ Failed to fetch transcript")

✅ Transcript fetched successfully!
📹 Video Title: LinkedIn Founder: How to Get Hired FAST While Others Get Rejected | Reid Hoffman
🔗 Video URL: https://www.youtube.com/watch?v=hBUTXCAc5zg
📝 Transcript Preview: layoffs, rescended offers, even entry level doesn't feel like entry level anymore. What's your way of navigating the AI world? >> Look, one of the piece of advice, especially for young people, is to s...
✅ Transcript saved to: transcripts/LinkedIn Founder How to Get Hired FAST While Others Get Rejected  Reid Hoffman.md
